# Lab 2：語音客服 Pipeline — STT → LLM → TTS

## 學習目標
1. 用 **Whisper** 把語音轉成文字（STT）
2. 用 **LLM** 扮演 HP 客服生成回答
3. 用 **TTS** 把回答轉回語音

## 流程
```
語音輸入 → Whisper → 文字問題 → LLM 客服 → 文字回答 → TTS → 語音輸出
```

In [ ]:
%%capture
!pip install openai langchain-openai

In [ ]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

client = OpenAI()
llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)
print("✓ 設定完成")

## Step 1：STT — 語音轉文字

先用 TTS 生成一段測試音訊（模擬使用者輸入），再用 Whisper 轉回文字。

In [ ]:
# 生成測試音訊（模擬使用者語音）
test_question = "請問 HP EliteBook 855 的電池續航力大概有多久？"

speech = client.audio.speech.create(
    model="tts-1", voice="alloy", input=test_question
)
speech.write_to_file("test_input.mp3")

# STT：用 Whisper 把音訊轉回文字
def speech_to_text(audio_path: str) -> str:
    with open(audio_path, "rb") as f:
        result = client.audio.transcriptions.create(model="whisper-1", file=f)
    return result.text

transcript = speech_to_text("test_input.mp3")
print(f"辨識結果：{transcript}")

## Step 2：LLM — 生成 HP 客服回答

LLM 要扮演 HP 客服，回答使用者的問題。  
**System prompt 決定了 LLM 的角色與回答風格。**

In [ ]:
# =============================================
# TODO：完成 HP 客服回答函數（約 5 行）
#
# 提示：
#   1. 定義 system_prompt，讓 LLM 扮演 HP 繁體中文客服
#   2. 用 llm.invoke([SystemMessage(...), HumanMessage(...)]) 呼叫
#   3. 回傳 response.content
# =============================================
def generate_answer(question: str) -> str:
    """讓 LLM 扮演 HP 客服回答問題"""
    pass  # 請實作這裡


# 快速測試
answer = generate_answer(transcript)
print(f"客服回答：{answer}")

## Step 3：TTS — 文字轉語音，完整測試

In [ ]:
# TTS：把回答轉成語音
def text_to_speech(text: str, output_path: str = "response.mp3") -> str:
    response = client.audio.speech.create(model="tts-1", voice="nova", input=text)
    response.write_to_file(output_path)
    return output_path

# 完整 Pipeline
print("=" * 50)
print(f"問：{transcript}")
print("=" * 50)
answer = generate_answer(transcript)
print(f"答：{answer}")
output_file = text_to_speech(answer)
print(f"\n✓ 語音已儲存至 {output_file}")